In [ ]:
#Импорт Библиотек
import requests
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import pearsonr

In [ ]:
# Чтение датасета
df = pd.read_excel("../data/vk_posts_clean.xlsx")
print(f"Размер: {df.shape}")
df.head(5)

In [ ]:
#Преобразуем дату публикации в формат datetime
df["Дата публикации"] = pd.to_datetime(df["Дата публикации"])
df["День недели"] = df["Дата публикации"].dt.dayofweek
df["День недели"] = df["Дата публикации"].dt.dayofweek
df["Час публикации"] = df["Дата публикации"].dt.hour

In [ ]:
#Корреляция между показателями
correlation_matrix = df[["Просмотры", "Лайки", "Комментарии"]].corr()

print("Корреляция между параметрами постов:")
print(correlation_matrix)

In [ ]:
#Корреляция по времени/просмотрам
df["Время публикации"] = pd.to_datetime(df["Дата публикации"]).dt.hour

correlation = df["Время публикации"].corr(df["Просмотры"])

print(f"Корреляция между временем публикации и просмотрами: {correlation:.2f}")

plt.figure(figsize=(8,5))
sns.scatterplot(x=df["Время публикации"], y=df["Просмотры"], alpha=0.5)
sns.lineplot(x=df["Время публикации"], y=df["Просмотры"], color="red", linewidth=2)
plt.xlabel("Время публикации")
plt.ylabel("Просмотры")
plt.title("Зависимость просмотров от времени публикации")
plt.xticks(range(0, 24))
plt.grid(True)

plt.show()


In [ ]:
#Средние просмотры по часам для каждого дня недели
views_by_hour_day = df.groupby(["Час публикации", "День недели"])["Просмотры"].mean().reset_index()
days_labels = ["Пн", "Вт", "Ср", "Чт", "Пт", "Сб", "Вс"]
colors = ["#FFDD44", "#FFBB33", "#FF8844", "#FF5522", "#FF2200", "#5599FF", "#0033CC"]

plt.figure(figsize=(10, 6))
for day in range(7):
    subset = views_by_hour_day[views_by_hour_day["День недели"] == day]
    sns.lineplot(x=subset["Час публикации"], y=subset["Просмотры"], marker="o", label=days_labels[day], color=colors[day])
plt.xlabel("Время публикации (часы)")
plt.ylabel("Средние просмотры")
plt.title("Зависимость просмотров от времени публикации по дням недели")
plt.xticks(range(0, 24))
plt.legend(title="День недели")
plt.grid()
plt.show()

In [ ]:
#Количество постов по дням недели
posts_by_day = df["День недели"].value_counts().reset_index()
posts_by_day.columns = ["День недели", "Количество постов"]
days_labels = {0: "Пн", 1: "Вт", 2: "Ср", 3: "Чт", 4: "Пт", 5: "Сб", 6: "Вс"}
posts_by_day["День недели"] = posts_by_day["День недели"].map(days_labels)
posts_by_day = posts_by_day.sort_values(by="День недели", key=lambda x: x.map({v: k for k, v in days_labels.items()}))

#Визуализация
plt.figure(figsize=(8, 5))
sns.barplot(data=posts_by_day, x="День недели", y="Количество постов", hue="День недели", palette="coolwarm", legend=False)
plt.xlabel("День недели")
plt.ylabel("Количество постов")
plt.title("Количество выложенных постов по дням недели")
plt.grid(axis="y")
plt.show()


#Среднее количество просмотров на пост по дням недели
avg_views_by_day = df.groupby("День недели")["Просмотры"].mean().reset_index()
days_labels = ["Пн", "Вт", "Ср", "Чт", "Пт", "Сб", "Вс"]
avg_views_by_day["День недели"] = days_labels

#Визуализация
plt.figure(figsize=(8, 5))
sns.barplot(data=avg_views_by_day, x="День недели", y="Просмотры", hue="День недели", palette="coolwarm", legend=False)
plt.xlabel("День недели")
plt.ylabel("Средние просмотры")
plt.title("Среднее количество просмотров по дням недели")
plt.grid(axis="y")
plt.show()

In [ ]:
#Корреляция по длинна поста/просмотрам
df["Длина текста"] = df["Текст поста"].astype(str).apply(len)  #Количество символов

plt.figure(figsize=(10, 6))
sns.regplot(x=df["Длина текста"], y=df["Просмотры"], scatter_kws={"alpha":0.5}, line_kws={"color":"red"})
plt.xlabel("Длина текста (символы)")
plt.ylabel("Просмотры")
plt.title("Зависимость просмотров от длины текста поста")
plt.grid()

corr, _ = pearsonr(df["Длина текста"], df["Просмотры"])
print(f"Корреляция между длиной текста и просмотрами: {corr:.2f}")

plt.show()

In [ ]:
#Просмотры по месяцам за весь период
df["Месяц"] = df["Дата публикации"].dt.to_period("M")
views_by_month = df.groupby("Месяц")["Просмотры"].sum().reset_index()

#"Месяц" в строку для красивого отображения на графике
views_by_month["Месяц"] = views_by_month["Месяц"].astype(str)

plt.figure(figsize=(14, 6))
sns.lineplot(x=views_by_month["Месяц"], y=views_by_month["Просмотры"], marker="o", color="b", linewidth=2)
plt.xticks(rotation=45, fontsize=8)
plt.xlabel("Месяц")
plt.ylabel("Количество просмотров")
plt.title("Динамика просмотров за всё время")
plt.grid()

plt.show()





#Последние 12 месяцев
last_year = df["Дата публикации"].max() - pd.DateOffset(years=1)
df_last_year = df[df["Дата публикации"] >= last_year]

df_last_year["Месяц"] = df_last_year["Дата публикации"].dt.to_period("M")
views_by_month = df_last_year.groupby("Месяц")["Просмотры"].sum().reset_index()

views_by_month["Месяц"] = views_by_month["Месяц"].astype(str)

plt.figure(figsize=(12, 6))
sns.lineplot(x=views_by_month["Месяц"], y=views_by_month["Просмотры"], marker="o", color="b", linewidth=2)
plt.xticks(rotation=45)
plt.xlabel("Месяц")
plt.ylabel("Количество просмотров")
plt.title("Динамика просмотров за последние 12 месяцев")
plt.grid()

plt.show()


In [ ]:
df.head()
print(df["Просмотры"].mean)
print(df["Лайки"].mean)